# Stochastic Optimization with pypsa2smspp

This notebook showcases an example on how to solve a PyPSA `StochasticNetwork` with SMS++.

To this goal, the following steps are performed:
1. Create a small PyPSA network
2. Optimizes it using PyPSA's built-in optimization capabilities to establish a baseline for comparison
3. Convert it to SMS++ same instance through the `pypsa2smspp` transformation pipeline
4. Optimizes the SMS++ model using the Two Stage Stochastic Block (ttsb_solver) solver from SMS++ tools
5. Compare the results obtained from both optimizations

This notebook will use the Two Stage Stochastic Block (ttsb_solver) solver from [SMS++ tools](https://gitlab.com/smspp/tools) to optimize the model. To do so, pypsa2smspp will convert the PyPSA model to an [TwoStageStochastic Block](https://gitlab.com/smspp/twostagestochasticblock) and optimize it using the default settings.

Note: it requires smspp-project version 0.6.0 or higher.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pypsa

from pypsa2smspp.transformation import Transformation
from pypsa2smspp.network_correction import add_slack_unit

## Settings

The example uses three equiprobable scenarios. Both demand and renewable availability are scenario-dependent.

In [ ]:
CASE_NAME = "stochastic_uc_example"

SOLVER_NAME = "gurobi"
SOLVER_OPTIONS = {
    "Threads": 32,
    "Method": 2,
    "Crossover": 0,
    "Seed": 123,
    "AggFill": 0,
    "PreDual": 0,
}

SCENARIO_PROBABILITIES = {
    "low": 1 / 3,
    "med": 1 / 3,
    "high": 1 / 3,
}

STOCHASTIC_PARAMETERS = ["demand", "renewables"]

## Build the deterministic base network

The network has two AC buses, one transmission line, two loads, two renewable generators, and one committable gas generator.

The gas generator makes this a unit-commitment example. Demand profiles are generated directly in the notebook.

In [ ]:
snapshots = pd.date_range("2025-01-01 00:00", periods=24, freq="h")

n = pypsa.Network()
n.set_snapshots(snapshots)

# Keep snapshot weights explicit.
n.snapshot_weightings.loc[:, :] = 1.0

# Carriers
n.add("Carrier", "AC")
n.add("Carrier", "solar")
n.add("Carrier", "onwind")
n.add("Carrier", "slack")

# Buses
n.add("Bus", "IT0 0", carrier="AC", v_nom=380)
n.add("Bus", "IT0 1", carrier="AC", v_nom=380)

# Transmission line
n.add(
    "Line",
    "Line 0-1",
    bus0="IT0 0",
    bus1="IT0 1",
    carrier="AC",
    x=0.1,
    r=0.01,
    s_nom=700,
)

# Loads
n.add("Load", "load_IT0_0", bus="IT0 0", carrier="AC")
n.add("Load", "load_IT0_1", bus="IT0 1", carrier="AC")

# Renewable generators
n.add(
    "Generator",
    "solar_IT0_0",
    bus="IT0 0",
    carrier="solar",
    p_nom=45,
    p_min_pu=0,
    p_max_pu=1,
    marginal_cost=0.01,
    p_nom_extendable=True,
    capital_cost=55
)

n.add(
    "Generator",
    "wind_IT0_1",
    bus="IT0 1",
    carrier="onwind",
    p_nom=35,
    p_min_pu=0,
    p_max_pu=1,
    marginal_cost=0.015,
    p_nom_extendable=True,
    capital_cost=70
)

# Deterministic base demand.
hours = np.arange(len(snapshots))

load_0 = 220 + 60 * np.sin(2 * np.pi * (hours - 7) / 24)
load_1 = 260 + 70 * np.sin(2 * np.pi * (hours - 8) / 24)
load_0 = np.maximum(load_0, 150)
load_1 = np.maximum(load_1, 180)

n.loads_t.p_set = pd.DataFrame(
    {
        "load_IT0_0": load_0,
        "load_IT0_1": load_1,
    },
    index=snapshots,
)

# Deterministic renewable availability.
solar_profile = np.maximum(0, np.sin(np.pi * (hours - 6) / 12))
wind_profile = 0.45 + 0.20 * np.sin(2 * np.pi * (hours + 3) / 24)
wind_profile = np.clip(wind_profile, 0.10, 0.85)

n.generators_t.p_max_pu = pd.DataFrame(
    {
        "solar_IT0_0": solar_profile,
        "wind_IT0_1": wind_profile,
    },
    index=snapshots,
)

# Add high-cost slack units to guarantee feasibility in all scenarios.
for b in range(len(n.buses)):
    n.add(
        "Generator",
        f"slack_{b}",
        bus=f"IT0 {b}",
        carrier="slack",
        marginal_cost=1000,
        p_nom=1e6,
    )

n

## Convert the network to a stochastic PyPSA network

The deterministic profiles are copied before calling `set_scenarios`. Then each scenario receives its own demand and renewable availability.

In [ ]:
base_load = n.loads_t.p_set.copy()
base_p_max_pu = n.generators_t.p_max_pu.copy()

n.set_scenarios(SCENARIO_PROBABILITIES)

# Scenario-dependent demand.
demand_multiplier = {
    "low": 0.85,
    "med": 1.00,
    "high": 1.20,
}

# Scenario-dependent renewable availability.
renewable_multiplier = {
    "low": 0.70,
    "med": 0.85,
    "high": 1.00,
}

for scenario in SCENARIO_PROBABILITIES:
    n.loads_t.p_set[scenario] = base_load * demand_multiplier[scenario]
    n.generators_t.p_max_pu[scenario] = (base_p_max_pu * renewable_multiplier[scenario]).clip(upper=1.0)

n

## Solve the stochastic unit-commitment problem with PyPSA

This is the PyPSA reference solution. The solve uses normal unit commitment, not linearized unit commitment.

In [ ]:
n_pypsa = n.copy()

n_pypsa.optimize(
    solver_name=SOLVER_NAME,
    solver_options=SOLVER_OPTIONS,
)

obj_pypsa = n_pypsa.objective + n_pypsa.objective_constant

statistics_pypsa = n_pypsa.statistics()

print(f"PyPSA objective: {obj_pypsa}")
statistics_pypsa

## Build and solve the SMS++ model

The transformation is configured for a Two-Stage Stochastic Block (`tssb`) with stochastic demand and renewable availability.

In [ ]:
n_smspp = n.copy()

transformation = Transformation(
    name=CASE_NAME,
    configfile="TSSBlock/TSSBSCfg_grb.txt",
    capacity_expansion_ucblock=True,
    stochastic_parameters={
        "stochastic_type": "tssb",
        "parameters": STOCHASTIC_PARAMETERS,
    },
)

smspp_network = transformation.create_model(n_smspp, verbose=True)

smspp_network.print_tree()

In [ ]:
transformation.optimize(verbose=True)

print(f"SMS++ objective: {transformation.result.objective_value}")

## Retrieve the SMS++ solution back into PyPSA

In [ ]:
n_smspp = transformation.retrieve_solution(n_smspp, verbose=True)

obj_smspp = transformation.result.objective_value
statistics_smspp = n_smspp.statistics()

relative_error_pct = (obj_smspp - obj_pypsa) / obj_pypsa * 100

print(f"PyPSA objective: {obj_pypsa}")
print(f"SMS++ objective: {obj_smspp}")
print(f"Relative error SMS++ vs PyPSA: {relative_error_pct:.6f}%")

statistics_smspp

## Print the summary

In [ ]:

summary = pd.DataFrame(
    [
        {
            "case": CASE_NAME,
            "objective_pypsa": obj_pypsa,
            "objective_smspp": obj_smspp,
            "relative_error_pct": relative_error_pct,
            "n_snapshots": len(n.snapshots),
            "n_scenarios": len(SCENARIO_PROBABILITIES),
            "stochastic_parameters": ", ".join(STOCHASTIC_PARAMETERS),
        }
    ]
)

summary